# 17 -- Checkpoint/resume validation: extend a real scan, compare against a fresh one

Validates the checkpointing scheme the way it's actually used in production.
Self-contained -- builds everything itself via `ptbuilder.ic.generate_surrogate`,
no round-tripping through other notebooks (notebook 15 has its own,
not-yet-migrated copy of this scan-building logic; this notebook doesn't
touch or depend on it).

**Workflow:**
1. Run **Section 1** to build two independent scans -- `gs1-4` and `gs1-5`
   (the ground truth) -- and write both their SLURM scripts. Submit both
   (they can run concurrently), wait for both to finish.
2. Run **Section 2** to extend the `gs1-4` scan to `gs1-5` in place (Format
   A -- each row's own embedded plateau checkpoint seeds it, no
   re-integration from `s=0`) and write its SLURM script. Submit, wait.
3. Run **Section 3** to compare a few representative rows between the
   extended scan and the independently-built ground truth at their shared
   `T_MAX`.

The ground-truth scan uses a distinct tag (`lb0.84_direct` vs `lb0.84`), so
it never collides with the extended scan's directory -- no rename-out-of-
the-way step needed.

The two scans' `wlist` grids differ (the extended scan keeps `gs1-4`'s
original frequency grid; the independently-built `gs1-5` scan has its own,
anchored to a different `GAMMA_STAR_MAX`) -- that's expected, not a bug.
Section 3 plots both curves together on the same axes rather than requiring
matching grids: if checkpointing is correct, they should trace the same
spectrum regardless of where each one happens to sample it.

In [ ]:
from pathlib import Path
from types import SimpleNamespace

import h5py
import numpy as np
import matplotlib.pyplot as plt

import ptbuilder as pt
from ptbuilder.ic import generate_surrogate
from ptbuilder.physics import Phi4Potential
from ptbuilder.weight_scan_diagnostics import load_kinematics

ROOT = Path(pt.__file__).parent.parent
LAMBDA_BAR = 0.84

kin = load_kinematics(ROOT / f'data/phi4_lambda_bar{LAMBDA_BAR:g}/instanton.h5')
model = SimpleNamespace(instanton=kin.profile, kinematics=kin,
                        potential=Phi4Potential(LAMBDA_BAR))

GAMMA_STAR_MIN = 1.0
KR_MIN, KR_MAX, N_OMEGA = 1.0, 100.0, 32   # production values

GAMMA_IJ_TARGETS = [1.0, 2.5, 5.0]

def scan_path(tag_label, gamma_star_max):
    tag = f'{tag_label}_gs{GAMMA_STAR_MIN:g}-{gamma_star_max:g}_kR{KR_MIN:g}-{KR_MAX:g}_nw{N_OMEGA}'
    return ROOT / 'data' / f'runtime_scan_{tag}'

## 1. Build gs1-4 and gs1-5 (independent, can run concurrently)

`gs1-4` uses `--overwrite` since whatever is on disk under that name
predates this session's checkpoint feature and needs regenerating with the
current binary (no `plateau_re` embedded otherwise). `gs1-5` is a normal
fresh build -- the ground truth Section 3 validates against.

In [ ]:
def write_slurm_script(scan, gpu_walltime='04:00:00', gpu_nodes=2, force_overwrite=False,
                       gpu_account='m3166_g'):
    """Writes scripts/runtime_scan_{tag}_gpu.sh for `scan` (from
    generate_surrogate) -- same xargs -P worker-pool pattern as notebook 15,
    duplicated here rather than shared since this is cluster-specific."""
    (ROOT / 'logs').mkdir(exist_ok=True)
    gpus_per_node = 4
    n_concurrent  = gpu_nodes * gpus_per_node
    n_tasks       = len(scan.manifest_rows)
    relative_manifest = scan.manifest.relative_to(ROOT)
    relative_output   = scan.output_root.relative_to(ROOT)
    overwrite_flag = ' --overwrite' if force_overwrite else ''

    script_path = ROOT / 'scripts' / f'runtime_scan_{scan.tag}_gpu.sh'
    script_path.write_text(f'''#!/bin/bash
#SBATCH --job-name=bm_{scan.tag}_gpu
#SBATCH --output=logs/bm_{scan.tag}_gpu_%j.out
#SBATCH --constraint=gpu
#SBATCH --qos=regular
#SBATCH -A {gpu_account}
#SBATCH --nodes={gpu_nodes}
#SBATCH --ntasks={n_concurrent}
#SBATCH --ntasks-per-node={gpus_per_node}
#SBATCH --cpus-per-task=32
#SBATCH --gpus-per-task=1
#SBATCH --time={gpu_walltime}
set -euo pipefail
REPO_ROOT="${{SLURM_SUBMIT_DIR:-$PWD}}"
cd "$REPO_ROOT"

run_one() {{
    local row_num=$1
    local line
    line=$(sed -n "$((row_num + 1))p" {relative_manifest})
    IFS=$'\t' read -r index gamma n_t setup name <<< "$line"
    output="{relative_output}/${{name}}"
    mkdir -p "$output"
    /usr/bin/time --verbose srun --exclusive --exact -N 1 -n 1 -c 32 --gpus-per-task=1 --cpu-bind=cores bin/bubblemaster_gpu "$setup" "$output/" --param {scan.n_min} --filon-panels-per-osc {scan.panels_per_oscillation} --s-batch-size {scan.s_batch_size}{overwrite_flag}
}}
export -f run_one

seq 1 {n_tasks} | xargs -P {n_concurrent} -I ROW bash -c 'run_one "$@"' _ ROW
''')
    script_path.chmod(0o755)
    print(f'Written: {script_path.relative_to(ROOT)}  '
          f'({n_tasks} tasks, {n_concurrent} concurrent slots on {gpu_nodes} nodes)'
          + (', --overwrite' if force_overwrite else ''))
    return script_path


scan_4        = generate_surrogate(model, ROOT, 'lb0.84', GAMMA_STAR_MIN, 4.0,
                                   KR_MIN, KR_MAX, N_OMEGA)
scan_5_direct = generate_surrogate(model, ROOT, 'lb0.84_direct', GAMMA_STAR_MIN, 5.0,
                                   KR_MIN, KR_MAX, N_OMEGA)

script_4        = write_slurm_script(scan_4, force_overwrite=True)
script_5_direct = write_slurm_script(scan_5_direct, force_overwrite=False)

print()
print('Submit both (can run concurrently):')
print(f'  sbatch {script_4.relative_to(ROOT)}')
print(f'  sbatch {script_5_direct.relative_to(ROOT)}')

## 2. Extend gs1-4 -> gs1-5

Run after both of Section 1's SLURM jobs finish. Renames the `gs1-4` scan
directory to `gs1-5`, extends its existing rows' `times[]` in place using
each row's own embedded plateau checkpoint, and computes any newly-in-range
rows fresh -- no `--overwrite` needed here, the GPU binary auto-detects
what's already done from `output_dir/` contents.

In [ ]:
scan_5_extended = generate_surrogate(model, ROOT, 'lb0.84', GAMMA_STAR_MIN, 5.0,
                                     KR_MIN, KR_MAX, N_OMEGA, extend_from=scan_4.scan_root)
script_5_extended = write_slurm_script(scan_5_extended, force_overwrite=False)
print(f'Submit: sbatch {script_5_extended.relative_to(ROOT)}')

## 3. Compare a few rows between the two gs1-5 scans

For each target `gamma_ij`, finds the nearest row in EACH scan independently
-- the two scans' `gamma_ij` grids are not bit-identical (the extended
scan's rows up to `gs1-4`'s old range are exactly `gs1-4`'s own rows; the
direct scan recomputes its own grid from scratch over the full `[1, 12.5]`
range), so this is a nearest-match, not an exact index match. Same
reasoning applies to `wlist`: the extended scan's rows keep their original
frequency grid (anchored to `GAMMA_STAR_MAX=4`), while the direct scan's
rows get one anchored to `GAMMA_STAR_MAX=5`. Both scans share the same
global `T_MAX` (fixed by `GAMMA_STAR_MAX=5` alone), so each row's LAST
result file is directly comparable in time even though the two rows aren't
exactly the same `gamma_ij` or `w` grid.

Reads scan paths from disk (not the in-memory `scan_*` objects from
Sections 1-2) -- this section is meant to be runnable on its own, e.g.
after a kernel restart while waiting on the cluster.

In [ ]:
DIRECT_SCAN   = scan_path('lb0.84_direct', 5.0)
EXTENDED_SCAN = scan_path('lb0.84', 5.0)

def load_manifest(scan_root):
    rows = []
    with open(scan_root / 'manifest.tsv') as f:
        next(f)   # header
        for line in f:
            index, gamma, n_t, setup, name = line.rstrip('\n').split('\t')
            rows.append(dict(index=int(index), gamma_ij=float(gamma), n_t=int(n_t),
                             setup=ROOT / setup, name=name))
    return rows

def nearest_row(rows, gamma_target):
    return min(rows, key=lambda r: abs(r['gamma_ij'] - gamma_target))

def load_last_result(scan_root, row):
    # t comes from the row's own setup.h5 times[], not the result file's "t"
    # attribute, for robustness against any scan generation on disk that
    # predates that attribute (see commit 528cecc) -- times[] is always there.
    last_i_t = row['n_t'] - 1
    path = scan_root / 'gpu' / row['name'] / f'result_{last_i_t:04d}.h5'
    with h5py.File(path) as f:
        w, spectrum = f['w'][:], f['spectrum'][:]
    with h5py.File(row['setup']) as f:
        t = float(f['times'][last_i_t])
    return w, spectrum, t

for path, label in [(DIRECT_SCAN, 'direct'), (EXTENDED_SCAN, 'extended')]:
    if not path.exists():
        raise FileNotFoundError(f'{path} not found -- see the workflow steps in the intro '
                                f'markdown cell ({label} scan missing)')

rows_direct   = load_manifest(DIRECT_SCAN)
rows_extended = load_manifest(EXTENDED_SCAN)

fig, axes = plt.subplots(1, len(GAMMA_IJ_TARGETS), figsize=(5*len(GAMMA_IJ_TARGETS), 4))
if len(GAMMA_IJ_TARGETS) == 1:
    axes = [axes]

for ax, gamma_target in zip(axes, GAMMA_IJ_TARGETS):
    row_d = nearest_row(rows_direct, gamma_target)
    row_e = nearest_row(rows_extended, gamma_target)

    if not (EXTENDED_SCAN / 'gpu' / row_e['name'] / f"result_{row_e['n_t']-1:04d}.h5").exists():
        raise FileNotFoundError(
            f"extended scan row gamma_ij={row_e['gamma_ij']:.3f} has no final result -- "
            f"wait for Section 2's SLURM job to finish")
    if not (DIRECT_SCAN / 'gpu' / row_d['name'] / f"result_{row_d['n_t']-1:04d}.h5").exists():
        raise FileNotFoundError(
            f"direct scan row gamma_ij={row_d['gamma_ij']:.3f} has no final result -- "
            f"wait for Section 1's SLURM job to finish")

    w_d, spec_d, t_d = load_last_result(DIRECT_SCAN, row_d)
    w_e, spec_e, t_e = load_last_result(EXTENDED_SCAN, row_e)

    assert abs(t_d - t_e) < 1e-6 * max(1.0, abs(t_d)), (
        f'direct t={t_d:.6f} != extended t={t_e:.6f} at gamma_ij~{gamma_target} -- '
        f'both scans should share the same global T_MAX')

    ax.plot(w_d, spec_d, 'o-', color='k', lw=1.5, ms=3,
           label=f"direct  (gamma_ij={row_d['gamma_ij']:.3f})")
    ax.plot(w_e, spec_e, 's--', color='tab:red', lw=1.5, ms=3,
           label=f"extended (gamma_ij={row_e['gamma_ij']:.3f})")
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('w')
    ax.set_title(f'gamma_ij~{gamma_target:g}  (t={t_d:.2f})')
    ax.legend(frameon=False, fontsize=8)

axes[0].set_ylabel('spectrum')
plt.tight_layout()
plt.savefig(ROOT / 'checkpoint_validation_scan_extend.pdf', bbox_inches='tight')
plt.show()